[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab16_eda_end_to_end/lab16_student.ipynb)

# Lab 16 — Exploratory Data Analysis End to End

### Digital Skills and Business Decisioning Using Python

This lab brings the whole course together. Instead of learning new tools, you'll use the ones you already have to carry out a complete **exploratory data analysis (EDA)** of the clothing reviews, from the first questions to written conclusions.

**Exploratory** means the goal is to *understand* the data: what's in it, what's wrong with it, how the variables behave, and which patterns are worth investigating further. An EDA doesn't prove causes or build models. It produces **evidence and good questions**, and it's always the first step before any serious modelling, such as the sentiment analysis and topic modelling of the follow-up course.

This notebook is also a **model for your final assignment**. Your assignment will follow the same steps on a different review dataset:

1. **Questions**: decide what you want to find out.
2. **Load and inspect** the data.
3. **Clean** it, and record every decision.
4. **Choose and describe the dependent variable.**
5. **Univariate analysis**: look at the main variables one at a time.
6. **Bivariate analysis**: compare the variables against the DV.
7. **Outliers and data problems**: find them, investigate them, decide what to do.
8. **Text vs. the DV**: what do unhappy customers write about?
9. **Conclusions**: findings, limitations, and next steps.

Notice how much of this notebook is **text**. In an EDA, the markdown cells that explain what you did, what you saw, and what it means are as important as the code. A chart without an interpretation is only half an answer.

At the end there are exercises that ask you to extend the analysis in your own direction.

### Setup

In [ ]:
import re
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/JanNikolaidovic/python-text-analytics-lab/main/data/womens_clothing_ecommerce_reviews.csv"

def load_reviews(url):
    """Load the clothing reviews and apply the cleaning steps from Lab 13."""
    reviews = pd.read_csv(url, index_col=0)
    reviews.columns = [name.lower().replace(" ", "_") for name in reviews.columns]
    reviews = reviews.rename(columns={"recommended_ind": "recommended",
                                      "division_name": "division",
                                      "department_name": "department"})
    reviews = reviews.dropna(subset=["review_text", "division", "department", "class_name"])
    reviews["title"] = reviews["title"].fillna("")
    reviews = reviews.drop_duplicates(subset=["review_text"])
    reviews["division"] = reviews["division"].replace({"Initmates": "Intimates"})
    reviews["recommended"] = reviews["recommended"].astype(bool)
    reviews["review_text"] = reviews["review_text"].str.replace(r"\s+", " ", regex=True).str.strip()

    # features from the text (Labs 09 and 13)
    reviews["n_words"] = reviews["review_text"].str.split().str.len()
    reviews["n_exclaims"] = reviews["review_text"].str.count("!")
    reviews["mentions_return"] = reviews["review_text"].str.contains(r"\breturn\w*", case=False)
    height = reviews["review_text"].str.extract(r"(\d)'\s?(\d{1,2})").astype(float)
    reviews["height_cm"] = ((height[0] * 12 + height[1]) * 2.54).round()
    reviews["sentiment"] = reviews["rating"].map({1: "negative", 2: "negative", 3: "neutral",
                                                  4: "positive", 5: "positive"})
    return reviews

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

BLUE = "#2a78d6"
ORDER = ["negative", "neutral", "positive"]
SENTIMENT_COLORS = {"negative": "#e34948", "neutral": "#85847e", "positive": "#2a78d6"}

The `load_reviews()` function is defined above but not called yet: in an EDA, you look at the raw data **before** cleaning it (Step 2).

---

### Step 1: Questions

Write your questions down **before** looking for answers. Otherwise it's easy to keep charting until something looks interesting and then call it a finding. Questions also keep the analysis focused on the DV.

For this dataset (an online store's clothing reviews), the business wants to understand **customer satisfaction**. Our questions:

* **Q1.** How satisfied are customers overall, and how is satisfaction distributed?
* **Q2.** Does satisfaction differ between product departments?
* **Q3.** Is satisfaction related to who the reviewer is (age) and how they describe themselves (mentioning their height)?
* **Q4.** How do negative reviews differ from positive ones in *how* they're written (length, exclamation marks, mentions of returns)?
* **Q5.** Which **words** are most characteristic of negative and of positive reviews?

Each question is specific enough that you can picture the table or chart that will answer it (the test from Lab 14).

---

### Step 2: Load and Inspect

First, the raw file, with no cleaning at all:

In [ ]:
raw = pd.read_csv(DATA_URL, index_col=0)
print(raw.shape)
raw.info()

In [ ]:
raw.describe()

**What we see.** 23,486 reviews with 10 columns: an ID, the reviewer's age, a title and review text, the star rating, a recommendation flag, a helpful-vote count, and three levels of product category. The text columns have missing values (about 16% of titles and 3.6% of reviews); the category columns miss 14 rows. From Lab 12 we also know about the `Initmates` typo. The numeric columns are complete and within sensible ranges (ages 18-99, ratings 1-5), but the helpful-vote count is very skewed (median 1, maximum 122).

---

### Step 3: Clean, and Record the Decisions

`load_reviews()` applies the cleaning steps from Lab 13 and adds the text features. In a report, list the cleaning decisions and how many rows each one affected, so a reader can judge whether they change the conclusions:

In [ ]:
df = load_reviews(DATA_URL)

print(f"Raw rows:     {len(raw)}")
print(f"Clean rows:   {len(df)}")
print(f"Removed:      {len(raw) - len(df)} ({(len(raw) - len(df)) / len(raw):.1%})")

| Decision | Rows affected | Reason |
|---|---|---|
| Dropped reviews with no text | 845 | Text analysis needs text. |
| Dropped rows with no category | 13 | Too few to matter; category unknown. |
| Dropped repeated review texts | 7 | Same text posted twice would double-count words. |
| Filled missing titles with `""` | ~2,960 | Title is optional; the review is still usable. |
| Fixed `Initmates` → `Intimates` | ~1,400 | Typo in the source data. |
| Collapsed whitespace in review text | ~2,000 | Line breaks and double spaces. |

In total we kept **96.3%** of the reviews. Could the cleaning bias the DV? Customers who leave a rating without any text turn out to be happier than average (about 70% of them give 5 stars, against 56% overall), so dropping them lowers the share of positive reviews very slightly: from 77.5% to 77.0%. That's too small to change any conclusion, but it's the kind of effect a report should mention. (Exercise 1 asks you to check it.)

---

### Step 4: The Dependent Variable (Q1)

Following Lab 14, the DV is **customer satisfaction**, measured by the star **rating**, and grouped into three **sentiment** classes for comparisons (negative = 1-2 stars, neutral = 3, positive = 4-5).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

rating_shares = df["rating"].value_counts(normalize=True).sort_index()
axes[0].bar(rating_shares.index, rating_shares.values, color=BLUE)
axes[0].set_title("Star ratings")
axes[0].set_xlabel("Star rating")
axes[0].set_ylabel("Share of reviews")

sentiment_shares = df["sentiment"].value_counts(normalize=True)[ORDER]
bars = axes[1].bar(sentiment_shares.index, sentiment_shares.values,
                   color=[SENTIMENT_COLORS[s] for s in ORDER])
axes[1].bar_label(bars, labels=[f"{v:.0%}" for v in sentiment_shares.values], padding=3)
axes[1].set_title("Sentiment classes (the DV)")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("")

fig.suptitle("Customers are mostly satisfied: 77% of reviews are positive", y=1.02)
plt.show()

In [ ]:
print(f"Mean rating:   {df['rating'].mean():.2f}")
print(f"Median rating: {df['rating'].median():.0f}")
print(f"Recommend:     {df['recommended'].mean():.1%}")

**Answer to Q1.** Customers are mostly satisfied: 55% of reviews give 5 stars, the median rating is 5, and 82% of reviewers recommend the product. Only about 10% of reviews are negative.

**What this means for the rest of the analysis.** The DV is **imbalanced**. Overall averages mostly describe satisfied customers, so from here on we always compare the three classes separately. With about 2,400 negative reviews, there's still plenty of data to study dissatisfaction.

We don't use `recommended` as a second DV: Lab 14 showed it's almost a copy of the rating (it only really varies at 3 stars).

---

### Step 5: Univariate Analysis

Before relating variables to the DV, look at each main variable on its own, in one grid of charts. `plt.subplots(2, 2)` returns a 2 × 2 grid of axes; `axes.flat` lets you loop through them in order.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ax_age, ax_words, ax_votes, ax_dept = axes.flat

sns.histplot(data=df, x="age", binwidth=3, color=BLUE, ax=ax_age)
ax_age.set_title("Age: peak in the late 30s, long right tail")
ax_age.set_xlabel("Age")

sns.histplot(data=df, x="n_words", binwidth=3, color=BLUE, ax=ax_words)
ax_words.set_title("Length: capped just above 100 words")
ax_words.set_xlabel("Words per review")

sns.histplot(data=df[df["positive_feedback_count"] <= 30], x="positive_feedback_count",
             binwidth=1, color=BLUE, ax=ax_votes)
ax_votes.set_title("Helpful votes (<= 30 shown): heavily skewed")
ax_votes.set_xlabel("Helpful votes")

dept_counts = df["department"].value_counts().sort_values()
ax_dept.barh(dept_counts.index, dept_counts.values, color=BLUE)
ax_dept.set_title("Departments: tops and dresses dominate")
ax_dept.set_xlabel("Number of reviews")

for ax in axes.flat:
    ax.set_ylabel("")
fig.tight_layout()
plt.show()

(`fig.tight_layout()` adjusts the spacing so the titles and labels of neighbouring charts don't overlap.)

**What we see.**

* **Age** is roughly bell-shaped around 40, with a right tail up to 99.
* **Review length** is spread out from 20 to 90 words and then piles up just under the website's 500-character limit. About one review in five hits the limit, so for long reviews the measured length is *truncated*.
* **Helpful votes** are heavily right-skewed: nearly half of the reviews have none. Use medians and shares, not means, to summarise them.
* **Departments** are very unequal: Tops and Dresses make up about 71% of the reviews, and Trend only 118. Any comparison involving Trend rests on few reviews.

---

### Step 6: Bivariate Analysis, Variables vs. the DV (Q2, Q3, Q4)

#### Q2: Satisfaction by department

In [ ]:
dept_summary = df.groupby("department").agg(
    reviews=("rating", "size"),
    avg_rating=("rating", "mean"),
    share_negative=("sentiment", lambda s: (s == "negative").mean()),
)
dept_summary.sort_values("share_negative", ascending=False).round(3)

(`lambda s: (s == "negative").mean()` is a custom aggregation: for each group, `s` is that group's `sentiment` column, and the lambda returns its share of negative reviews. `.agg()` accepts any function like this, in place of a name like `"mean"`.)

In [ ]:
shares = pd.crosstab(df["department"], df["sentiment"], normalize="index")[ORDER].sort_values("negative")

fig, ax = plt.subplots(figsize=(9, 4))
shares.plot(kind="barh", stacked=True, color=[SENTIMENT_COLORS[s] for s in ORDER],
            edgecolor="white", linewidth=1.5, width=0.75, ax=ax)
ax.set_title("Trend stands out; the other departments are similar")
ax.set_xlabel("Share of reviews")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="Sentiment", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

**Answer to Q2.** Five of the six departments are very similar: 9-11% negative reviews and average ratings between 4.14 and 4.28. Dresses and Tops are slightly worse than Bottoms, Intimate, and Jackets. **Trend** is the exception, with 18% negative reviews and an average of 3.84, but it's also the smallest department (118 reviews), so the result should be reported with that caveat. Trend items are, by definition, fashion-forward and more experimental, which may explain more mixed reactions; the data can't tell us that for sure.

#### Q3: Satisfaction and the reviewer

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[17, 29, 39, 49, 59, 69, 100],
                         labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70+"])
df["mentions_height"] = df["height_cm"].notna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.pointplot(data=df, x="age_group", y="rating", color=BLUE, ax=axes[0])
axes[0].set_ylim(1, 5)
axes[0].set_title("Average rating barely changes with age")
axes[0].set_xlabel("Age group")
axes[0].set_ylabel("Average rating (1-5)")

sns.barplot(data=df, x="sentiment", y="mentions_height", order=ORDER,
            hue="sentiment", palette=SENTIMENT_COLORS, legend=False, ax=axes[1])
axes[1].set_title("Satisfied reviewers mention their height more often")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Share of reviews mentioning a height")

plt.show()

**Answer to Q3.** **Age makes almost no difference**: every age group averages between 4.1 and 4.3 stars (shown on the full 1-5 scale, so the chart doesn't exaggerate small differences). **Mentioning a height does**: about 19% of positive reviews mention the reviewer's height, against about 13% of negative ones. A likely explanation is that satisfied customers write *advice for other shoppers* ("I'm 5'4 and the XS fits perfectly"), while dissatisfied customers focus on what went wrong. In Lab 09, with only 600 reviews, this difference was too small to see reliably: a reminder that sample size matters.

#### Q4: How the reviews are written

In [ ]:
writing = df.groupby("sentiment").agg(
    median_words=("n_words", "median"),
    avg_exclaims=("n_exclaims", "mean"),
    share_return=("mentions_return", "mean"),
    share_3plus_votes=("positive_feedback_count", lambda v: (v >= 3).mean()),
).loc[ORDER]
writing.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = [("avg_exclaims", "Average exclamation marks"),
           ("share_return", "Share mentioning a return"),
           ("share_3plus_votes", "Share with 3+ helpful votes")]

for ax, (column, label) in zip(axes, metrics):
    ax.bar(writing.index, writing[column], color=[SENTIMENT_COLORS[s] for s in ORDER])
    ax.set_title(label)
    ax.set_xlabel("Sentiment")

fig.suptitle("Negative reviews: fewer exclamation marks, more returns, more helpful votes", y=1.03)
plt.show()

**Answer to Q4.**

* **Length**: negative reviews are not longer in general; neutral reviews are the longest (median 64 words vs 60 negative and 58 positive). Customers with mixed feelings explain both sides.
* **Exclamation marks**: positive reviews use about twice as many (0.73 vs 0.37 per review).
* **Returns**: 21% of negative reviews mention returning the item, compared with 4% of positive ones: a five-fold difference, and the clearest writing-related signal in the data.
* **Helpful votes**: about 35% of negative reviews get 3 or more helpful votes, against 25% of positive reviews. Shoppers value detailed warnings.

---

### Step 7: Outliers and Data Problems

An **outlier** is a value far from the rest. It can be a **data error** (to fix or remove) or a **real but unusual case** (to keep, and maybe to learn from). You can't tell which from the number alone: you have to look.

A standard way to flag candidates is the **IQR rule**. The **interquartile range** (IQR) is the distance between the 25% and 75% quartiles, the middle half of the data. Values more than 1.5 × IQR below the 25% quartile or above the 75% quartile are flagged. (This is the same rule a box plot uses to draw its dots.)

In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

low, high = iqr_bounds(df["height_cm"])
flagged = df[(df["height_cm"] < low) | (df["height_cm"] > high)]

print(f"Height bounds: {low:.0f} to {high:.0f} cm")
print(f"Flagged: {len(flagged)} of {df['height_cm'].notna().sum()} reviews with a height")
flagged["height_cm"].sort_values().values

Some flagged values (188 and 190 cm) are unusual but perfectly possible heights: those are **real outliers**, and we keep them. But 61 cm or 307 cm is impossible. Look at the text around the match to see what went wrong:

In [ ]:
implausible = df[(df["height_cm"] < 140) | (df["height_cm"] > 200)]

for text in implausible["review_text"]:
    match = re.search(r"(\d)'\s?(\d{1,2})", text)
    start = max(0, match.start() - 25)
    print(f"{match.group()!r:>8}  ...{text[start:match.end() + 20]}...")

Now it's clear. These are **parsing errors**, not strange customers. Reviewers wrote heights in forms the Lab 09 pattern didn't expect: `5' 21/2"` (5 feet 2½ inches, read as 5 feet 21 inches), or with the symbols reversed (`5"7'`, read as 7 feet 14 inches). The decision: treat heights outside a plausible adult range (140-200 cm) as missing, and note it in the report.

In [ ]:
plausible = df["height_cm"].between(140, 200)
df["height_cm"] = df["height_cm"].where(plausible)

print(df["height_cm"].describe().round(1))

`.where(condition)` keeps the values where the condition is `True` and replaces the rest with `NaN`. Six values were removed; the remaining heights range from 147 to 190 cm, with a median of 165 cm.

**The helpful votes** tell a different story:

In [ ]:
low, high = iqr_bounds(df["positive_feedback_count"])
share_flagged = (df["positive_feedback_count"] > high).mean()
print(f"Upper bound: {high}, flagged: {share_flagged:.1%} of reviews")

The IQR rule flags about 9.5% of reviews as outliers, but nothing is wrong with them: some reviews genuinely get many votes. For a heavily skewed variable, the rule mostly flags the long tail. **We keep all of them**, and we summarise votes with medians and shares instead of means (as in Q4).

**Review length** has no outliers to speak of: the 500-character cap cuts off the long end. That's a data limitation, not an outlier problem, and it goes into the conclusions.

---

### Step 8: Text vs. the DV (Q5)

Lab 15 showed that the **most frequent** words are similar across sentiment classes: everybody talks about size, fit, and fabric. To find what's **characteristic** of each class, compare how often each word is used in negative vs. positive reviews:

1. For each review, take its **set** of words (so a word counts once per review).
2. For each word, compute the **share of negative reviews** using it and the **share of positive reviews** using it.
3. Divide the two shares. A ratio of 5 means the word is five times as common (per review) in negative reviews.
4. Only keep words used in at least 100 reviews, so the ratios aren't based on a handful of uses.

In [ ]:
from collections import Counter

def word_set(text):
    tokens = {t.strip("'") for t in re.findall(r"[a-z']+", text.lower())}
    return tokens - {""}

negative_texts = df.loc[df["sentiment"] == "negative", "review_text"]
positive_texts = df.loc[df["sentiment"] == "positive", "review_text"]

negative_counts = Counter()
for text in negative_texts:
    negative_counts.update(word_set(text))

positive_counts = Counter()
for text in positive_texts:
    positive_counts.update(word_set(text))

words = pd.DataFrame({"negative_reviews": pd.Series(negative_counts),
                      "positive_reviews": pd.Series(positive_counts)}).fillna(0)
words = words[words["negative_reviews"] + words["positive_reviews"] >= 100].copy()

words["negative_share"] = words["negative_reviews"] / len(negative_texts)
words["positive_share"] = words["positive_reviews"] / len(positive_texts)
words["ratio"] = (words["negative_share"] + 0.001) / (words["positive_share"] + 0.001)

print(f"{len(words)} words used in at least 100 reviews")
words.sort_values("ratio", ascending=False).head(10).round(3)

Two small details:

* Building a DataFrame from two `Counter`s (turned into Series) lines the words up automatically by their index; a word missing from one group gets `NaN`, which `fillna(0)` turns into a count of 0.
* Adding 0.001 to both shares avoids dividing by zero for words that never appear in positive reviews. Note also that stopwords don't need removing here: words like `the` are used equally by both groups, so their ratio is close to 1 and they never reach the top.

Now chart the 15 most characteristic words of each class:

In [ ]:
most_negative = words.sort_values("ratio", ascending=False).head(15)
most_positive = words.sort_values("ratio").head(15)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

axes[0].barh(most_negative.index[::-1], most_negative["ratio"][::-1], color=SENTIMENT_COLORS["negative"])
axes[0].set_title("Most characteristic of negative reviews")
axes[0].set_xlabel("Times more common in negative than positive reviews")

axes[1].barh(most_positive.index[::-1], (1 / most_positive["ratio"])[::-1], color=SENTIMENT_COLORS["positive"])
axes[1].set_title("Most characteristic of positive reviews")
axes[1].set_xlabel("Times more common in positive than negative reviews")

fig.tight_layout()
plt.show()

(`[::-1]` reverses the order so the strongest word is drawn at the top. For the positive side, `1 / ratio` expresses the ratio the other way round, so both panels read as "times more common in this group".)

**Answer to Q5.** The words tell a clear story.

* **Negative reviews** are about **quality and cut**: `cheap`, `unflattering`, `huge`, `sack`, `tent`, `awkward`, `bad` (the garment is badly made or hangs shapelessly), about **disappointment**: `disappointed`, `unfortunately`, `sadly`, `sad`, `excited` (*"I was so excited to get this..."*), and about the outcome: `returned`, `returning`.
* **Positive reviews** are about **wearing and styling** the item: `dressed` (up or down), `compliments`, `versatile`, `paired`, `booties`, `boots`, `heels`, `flats`, `sandals`, `cooler` (*cooler days*), and about **contentment**: `highly` (recommend), `pleased`, `keeper`. Even `complaint` makes the list, as in *"my only complaint is..."*: happy customers mention small flaws in passing.

That's a genuinely useful business insight from a few lines of counting: unhappy customers talk about the *product*, while happy customers talk about their *life with the product*.

**A caution.** Single words ignore context. `excited` is characteristic of *negative* reviews because it usually appears in disappointed sentences. Counting words cannot distinguish "not cheap" from "cheap". The follow-up course introduces methods that take more context into account.

---

### Step 9: Conclusions

**Main findings**

1. **Customers are mostly satisfied.** 77% of reviews are positive (4-5 stars), 12% neutral, and 10% negative; 82% of reviewers recommend the product.
2. **Satisfaction is similar across departments, except Trend**, which has nearly twice the share of negative reviews (18% vs 9-11%), based on a small group (118 reviews).
3. **Who the reviewer is barely matters**: average ratings are nearly identical across age groups.
4. **How reviews are written differs clearly by sentiment.** Negative reviews mention returning the item five times as often, use half as many exclamation marks, mention the reviewer's height less often, and are more often voted helpful by other customers.
5. **The words differ in a meaningful way.** Negative reviews focus on poor quality and shapeless cuts (`cheap`, `unflattering`, `huge`, `sack`) and on returns; positive reviews focus on styling and wearing the item (`dressed`, `compliments`, `versatile`, `paired`).

**Limitations**

* **Who writes reviews.** Reviewers choose to write; they aren't a random sample of customers. The true satisfaction of all customers may differ.
* **Truncated text.** About 20% of reviews hit the 500-character limit, so length and word counts are cut short for the longest reviews.
* **Simple text features.** Word counts and regex patterns ignore context and negation, and a few patterns misfire (the height parsing errors in Step 7).
* **Description, not causes.** Every finding is an association. Exclamation marks don't cause satisfaction, and Trend items may be rated lower because of the kind of customer who buys them rather than the items themselves.
* **Missing information.** The data has no dates, prices, or product names, which limits the questions we can ask.

**Next steps**

* Test whether the Trend difference holds up with a statistical test, or with more data.
* Look at the 3-star (neutral) reviews in detail: they are where recommendation is most divided.
* Go beyond single words: **sentiment analysis** of the text itself (does the text agree with the stars?) and **topic modelling** (what themes run through the negative reviews?). Both are topics of the follow-up course on text analytics.

---

### Your Turn

**Exercise 1: Did Cleaning Bias the DV? (easy)**

Step 3 claimed that dropping reviews without text doesn't shift the rating distribution. Check it: using `raw`, compute the share of each rating among **all** rows and among the rows where `Review Text` is missing, and display both side by side in one DataFrame (hint: `pd.DataFrame({"all": ..., "missing_text": ...})`). Are they similar?

In [ ]:
# TODO: compare the rating distribution of all raw rows with that of rows missing the review text

In [ ]:
# @solution
comparison = pd.DataFrame({
    "all": raw["Rating"].value_counts(normalize=True).sort_index(),
    "missing_text": raw.loc[raw["Review Text"].isna(), "Rating"].value_counts(normalize=True).sort_index(),
})
comparison.round(3)
# Reviews without text are clearly more positive (70% five-star vs 56% overall).
# But they are only 3.6% of rows, so dropping them lowers the positive share by
# only about half a percentage point (77.5% -> 77.0%). Worth a sentence in the report.

**Exercise 2: Characteristic Words of Neutral Reviews (medium)**

Repeat Step 8, but compare **neutral** reviews with **positive** ones. Which words are most characteristic of 3-star reviews? Chart the top 15 and write two or three sentences interpreting them.

In [ ]:
# TODO: characteristic words of neutral vs positive reviews, chart the top 15, interpret

In [ ]:
# @solution
neutral_texts = df.loc[df["sentiment"] == "neutral", "review_text"]
neutral_counts = Counter()
for text in neutral_texts:
    neutral_counts.update(word_set(text))

compare = pd.DataFrame({"neutral_reviews": pd.Series(neutral_counts),
                        "positive_reviews": pd.Series(positive_counts)}).fillna(0)
compare = compare[compare["neutral_reviews"] + compare["positive_reviews"] >= 100].copy()
compare["ratio"] = ((compare["neutral_reviews"] / len(neutral_texts) + 0.001)
                    / (compare["positive_reviews"] / len(positive_texts) + 0.001))
top_neutral = compare.sort_values("ratio", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top_neutral.index[::-1], top_neutral["ratio"][::-1], color=SENTIMENT_COLORS["neutral"])
ax.set_title("Most characteristic of neutral (3-star) reviews")
ax.set_xlabel("Times more common in neutral than positive reviews")
plt.show()
# Neutral reviews share much of the negative vocabulary (disappointment, returns, shapeless fits),
# but in weaker doses: they describe items that were close to working but didn't quite.

**Exercise 3: Your Own Question (medium)**

Add one more question to Step 1 (for example about `class_name`, `division`, title words, or reviews mentioning `petite`), and answer it the way this notebook does: a question, a table or chart, and a short written answer that includes numbers and at least one caveat.

In [ ]:
# TODO: your question, the analysis, and a written answer

In [ ]:
# @solution
# Question: do reviews that mention petite sizing have a different sentiment mix?
df["mentions_petite"] = df["review_text"].str.contains(r"\bpetite\b", case=False)

petite = pd.crosstab(df["mentions_petite"], df["sentiment"], normalize="index")[ORDER]
print(petite.round(3))

# Answer: reviews that mention "petite" are slightly less often negative (8.0% vs 10.7%)
# and slightly more often positive (79.9% vs 76.8%).
# Caveat: "petite" can describe the customer, the size ordered, or the product line,
# so this simple keyword doesn't separate those meanings.

---

**For your final assignment**, use this notebook's structure as your template: the nine steps, a markdown cell after every result that says what you see and what it means, a table of cleaning decisions, and conclusions that list findings, limitations, and next steps. The dataset will be different, so your questions, your DV, and your findings will be too.